In [1]:
!pip install wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 4.6 MB/s eta 0:00:00


In [2]:
import math
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, classification_report, average_precision_score, hamming_loss, accuracy_score
from torch.amp import autocast

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class AdvancedECGTransformer(nn.Module):
    def __init__(self, in_channels=12, d_model=256, nhead=8, num_classes=5, num_layers=6):
        super().__init__()
        self.d_model = d_model # Saved for scaling
        
        self.stem = nn.Sequential(
            nn.Conv1d(in_channels, d_model // 2, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(d_model // 2),
            nn.GELU(),
            nn.Conv1d(d_model // 2, d_model, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(d_model),
            nn.GELU()
        )
        self.pos_encoder = PositionalEncoding(d_model=d_model)
        
        # --- FIX 2: norm_first=True prevents the gradients from dying ---
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4, 
            dropout=0.1, activation='gelu', batch_first=True,
            norm_first=True 
        )
        # We must add a final LayerNorm if norm_first=True
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, norm=nn.LayerNorm(d_model)
        )
        
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(0.3), nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x):
        x = self.stem(x) 
        x = x.transpose(1, 2) 
        
        # --- FIX 3: Scale the CNN features so Positional Encodings don't drown them ---
        x = x * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        
        x = self.transformer(x)
        
        # --- FIX 1: Global Average Pooling routes gradients to all tokens smoothly ---
        x_avg = torch.mean(x, dim=1)
        
        return self.head(x_avg)

# Evaluation function remains unchanged
def evaluate_model_advanced(model, dataloader, device):
    model.eval()
    all_probs, all_targets = [], []
    
    with torch.no_grad():
        for signals, labels in dataloader:
            signals = signals.to(device, non_blocking=True)
            with autocast('cuda', dtype=torch.bfloat16):
                probs = torch.sigmoid(model(signals)) 
            all_probs.append(probs.cpu().float().numpy())
            all_targets.append(labels.numpy())
            
    all_probs = np.vstack(all_probs)
    all_targets = np.vstack(all_targets)
    
    aucs, auprcs = [], []
    superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
    
    for i in range(len(superclasses)):
        if len(np.unique(all_targets[:, i])) == 2:
            aucs.append(roc_auc_score(all_targets[:, i], all_probs[:, i]))
            auprcs.append(average_precision_score(all_targets[:, i], all_probs[:, i]))
            
    macro_auc = np.mean(aucs) if len(aucs) > 0 else 0.0
    macro_auprc = np.mean(auprcs) if len(auprcs) > 0 else 0.0
        
    return macro_auc, macro_auprc

In [3]:
import os
import ast
import wfdb
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

class PTBXLDataset(Dataset):
    def __init__(self, data_path, metadata_csv, scp_csv, sampling_rate=100, folds=None):
        super().__init__()
        self.data_path = data_path
        self.sampling_rate = sampling_rate
        
        self.df = pd.read_csv(metadata_csv, index_col='ecg_id')
        self.df['scp_codes'] = self.df.scp_codes.apply(lambda x: ast.literal_eval(x))
        
        # Filter by Folds
        if folds is not None:
            self.df = self.df[self.df.strat_fold.isin(folds)]
            print(f"Filtered dataset to folds: {folds}. Records: {len(self.df)}")
        
        self.scp_statements = pd.read_csv(scp_csv, index_col=0)
        self.diagnostic_df = self.scp_statements[self.scp_statements.diagnostic == 1]
        
        self.superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
        self.labels = self._get_labels()
        self.signals = []
        
        print(f"Pre-loading {sampling_rate}Hz dataset into RAM...")
        for idx in tqdm(range(len(self.df)), desc="Loading ECGs"):
            rel_path = self.df.iloc[idx].filename_hr if self.sampling_rate == 500 else self.df.iloc[idx].filename_lr 
            full_path = os.path.join(self.data_path, rel_path)
            
            record = wfdb.rdrecord(full_path)
            signal = np.nan_to_num(record.p_signal)
            signal_tensor = torch.tensor(signal, dtype=torch.float32).transpose(0, 1)
            
            self.signals.append(signal_tensor)
        
    def _get_labels(self):
        def aggregate_diagnostic(y_dict):
            tmp = []
            for key in y_dict.keys():
                if key in self.diagnostic_df.index:
                    superclass = self.diagnostic_df.loc[key].diagnostic_class
                    if pd.notna(superclass):
                        tmp.append(superclass)
            return list(set(tmp))

        self.df['diagnostic_superclass'] = self.df.scp_codes.apply(aggregate_diagnostic)
        
        labels = np.zeros((len(self.df), len(self.superclasses)))
        for i, classes in enumerate(self.df['diagnostic_superclass']):
            for c in classes:
                if c in self.superclasses:
                    idx = self.superclasses.index(c)
                    labels[i, idx] = 1
        return torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.signals[idx], self.labels[idx]

In [4]:
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# 1. Setup Directories
BASE_DIR = "/kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"

# 2. Load Train (Folds 1-8) and Validation (Fold 9)
train_dataset = PTBXLDataset(
    data_path=BASE_DIR, metadata_csv=os.path.join(BASE_DIR, "ptbxl_database.csv"),
    scp_csv=os.path.join(BASE_DIR, "scp_statements.csv"), sampling_rate=100,
    folds=[1, 2, 3, 4, 5, 6, 7, 8]
)

val_dataset = PTBXLDataset(
    data_path=BASE_DIR, metadata_csv=os.path.join(BASE_DIR, "ptbxl_database.csv"),
    scp_csv=os.path.join(BASE_DIR, "scp_statements.csv"), sampling_rate=100,
    folds=[9] # Fold 9 is the standard PTB-XL validation set
)

# num_workers=0 remains active to protect Kaggle RAM
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

# 3. Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the Transformer baseline
model = AdvancedECGTransformer(
    in_channels=12, d_model=256, nhead=8, num_classes=5, num_layers=6
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)
criterion = nn.BCEWithLogitsLoss()

# 4. The Training Loop
epochs = 30
max_norm = 1.0

print("Starting Transformer Baseline Training on PTB-XL...")
for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    valid_batches = 0
    
    for i, (signals, labels) in enumerate(train_dataloader):
        signals, labels = signals.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True) 
        
        with autocast('cuda', dtype=torch.bfloat16):
            outputs = model(signals)
            loss = criterion(outputs, labels)
            
        if torch.isnan(loss):
            continue
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        
        optimizer.step()
        total_loss += loss.item()
        valid_batches += 1
        
    scheduler.step()
    avg_train_loss = total_loss / max(valid_batches, 1)
    
    # Validation check
    val_auc, val_auprc = evaluate_model_advanced(model, val_dataloader, device)
    
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val AUC: {val_auc:.4f} | Val AUPRC: {val_auprc:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")

# Save the weights
torch.save(model.state_dict(), '/kaggle/working/transformer_ptbxl_weights.pth')
print("Transformer Model saved successfully!")

Filtered dataset to folds: [1, 2, 3, 4, 5, 6, 7, 8]. Records: 17441
Pre-loading 100Hz dataset into RAM...


Loading ECGs:   0%|          | 0/17441 [00:00<?, ?it/s]

Filtered dataset to folds: [9]. Records: 2193
Pre-loading 100Hz dataset into RAM...


Loading ECGs:   0%|          | 0/2193 [00:00<?, ?it/s]

/tmp/ipykernel_58/2806701158.py:43: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Starting Transformer Baseline Training on PTB-XL...
Epoch [1/30] | Train Loss: 0.3695 | Val AUC: 0.8970 | Val AUPRC: 0.7526 | LR: 0.000293
Epoch [2/30] | Train Loss: 0.3013 | Val AUC: 0.9082 | Val AUPRC: 0.7692 | LR: 0.000271
Epoch [3/30] | Train Loss: 0.2795 | Val AUC: 0.9146 | Val AUPRC: 0.7815 | LR: 0.000238
Epoch [4/30] | Train Loss: 0.2630 | Val AUC: 0.9175 | Val AUPRC: 0.7895 | LR: 0.000197
Epoch [5/30] | Train Loss: 0.2461 | Val AUC: 0.9179 | Val AUPRC: 0.7846 | LR: 0.000150
Epoch [6/30] | Train Loss: 0.2308 | Val AUC: 0.9180 | Val AUPRC: 0.7860 | LR: 0.000104
Epoch [7/30] | Train Loss: 0.2137 | Val AUC: 0.9167 | Val AUPRC: 0.7833 | LR: 0.000063
Epoch [8/30] | Train Loss: 0.1966 | Val AUC: 0.9195 | Val AUPRC: 0.7892 | LR: 0.000030
Epoch [9/30] | Train Loss: 0.1823 | Val AUC: 0.9174 | Val AUPRC: 0.7843 | LR: 0.000008
Epoch [10/30] | Train Loss: 0.1762 | Val AUC: 0.9178 | Val AUPRC: 0.7858 | LR: 0.000300
Epoch [11/30] | Train Loss: 0.2272 | Val AUC: 0.9154 | Val AUPRC: 0.7796 | LR